# Can we predict next-month churn?
Target: `churn_next_month`. Data: monthly account snapshots (3,000 accounts, 2024-01 → 2025-11).

Plan: audit the data (leakage, time structure), build a leakage-free baseline and a gradient-boosted model, and evaluate with a **time-based split** (that is how the model would be used).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); df['month']=pd.PeriodIndex(df.snapshot_month,freq='M')
print(df.shape, 'accounts:',df.account_ref.nunique(), 'dupe account-months:',df.duplicated(['account_ref','snapshot_month']).sum())
print('missing:',df.isna().sum().sum(),'| churn rate:',round(df.churn_next_month.mean(),3))
df.head()

## 1. Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export of 2026-01-05* – i.e. computed after the snapshot, using knowledge of when the account ended. Compare it with `tenure_months`.

In [ ]:
df['gap']=df.lifetime_months-df.tenure_months
print(df.groupby('churn_next_month').gap.describe())
print('P(churn | lifetime==tenure) =',df[df.gap==0].churn_next_month.mean(), '| P(churn | lifetime>tenure) =',df[df.gap>0].churn_next_month.mean())

`lifetime_months == tenure_months` **exactly** when the account churns next month (gap is 0 for every churner, ≥1 for every non-churner). It encodes the outcome and would not be available at prediction time, so it is **excluded** from all models. The demo below shows how misleading it would be.

In [ ]:
feat_all=['plan','seats','monthly_fee','tenure_months','usage_hours','support_tickets']
def prep(d,extra=()):
    X=d[feat_all+list(extra)].copy(); X['plan']=X.plan.map({'basic':0,'pro':1,'team':2}); return X
tr_=df.month<=pd.Period('2025-05'); te_=df.month>=pd.Period('2025-07')   # 1-month gap: label of month m looks at m+1
leaky=HistGradientBoostingClassifier(random_state=0).fit(prep(df[tr_],['lifetime_months']),df[tr_].churn_next_month)
print('AUC WITH leaky lifetime_months (do not trust):',round(roc_auc_score(df[te_].churn_next_month,leaky.predict_proba(prep(df[te_],['lifetime_months']))[:,1]),3))

## 2. Exploration
Churn is strongly seasonal – it spikes for snapshots in Oct 2024–Feb 2025.

In [ ]:
g=df.groupby('month').agg(n=('churn_next_month','size'),churn=('churn_next_month','mean'))
fig,ax=plt.subplots(1,2,figsize=(13,3.5))
g.churn.plot(ax=ax[0],marker='o',title='Churn rate by snapshot month'); g.n.plot.bar(ax=ax[1],title='Snapshots per month')
plt.tight_layout(); plt.show()
print(df.groupby('plan')[['seats','monthly_fee','usage_hours','support_tickets','churn_next_month']].mean().round(3))
for col in ['tenure_months','support_tickets']:
    print(df.groupby(df[col].clip(upper=12 if col=='tenure_months' else 4)).churn_next_month.agg(['mean','size']).round(3).T)
df['usage_bin']=pd.qcut(df.usage_hours,8); print(df.groupby('usage_bin',observed=True).churn_next_month.mean().round(3))

## 3. Modelling
* **Split:** train on snapshots ≤ 2025-05, test on snapshots ≥ 2025-07 (June dropped as a gap, since labels look one month ahead). Test churn is ~8–11 %.
* **Seasonality caveat:** the data contain only one winter, so the model cannot learn a calendar effect that would generalise; I use **no calendar features**. The train period includes the abnormal winter-2024/25 churn spike, which may dilute what the model learns about normal months.
* Metrics: ROC-AUC and average precision (PR-AUC; base rate ≈ test churn rate) – accuracy is uninformative at this imbalance.

In [ ]:
train,test=df[tr_],df[te_]
Xtr,ytr,Xte,yte=prep(train),train.churn_next_month,prep(test),test.churn_next_month
print('train',len(train),'churn',round(ytr.mean(),3),'| test',len(test),'churn',round(yte.mean(),3))
print('test rows whose account is also in train:',round(test.account_ref.isin(train.account_ref).mean(),2))
models={'logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
        'grad. boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)}
res={}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; res[n]=p
    print(f'{n:15s} AUC={roc_auc_score(yte,p):.3f}  AP={average_precision_score(yte,p):.3f}')
print(f'{"base rate":15s} AP={yte.mean():.3f}')

Uncertainty: bootstrap the test set by **account** (rows of one account are correlated).

In [ ]:
rng=np.random.default_rng(0); acc=test.account_ref.values; ua=np.unique(acc); idx=pd.Series(np.arange(len(test))).groupby(acc).apply(list)
out={n:[] for n in res}
for _ in range(300):
    ii=np.concatenate([idx[a] for a in rng.choice(ua,len(ua))])
    for n,p in res.items(): out[n].append(roc_auc_score(yte.values[ii],p[ii]))
for n,v in out.items(): print(n,'AUC 95% CI',np.round(np.percentile(v,[2.5,97.5]),3))

## 4. What drives it, and is it actionable?

In [ ]:
gb=models['grad. boosting']
pi=permutation_importance(gb,Xte,yte,scoring='roc_auc',n_repeats=10,random_state=0)
print(pd.Series(pi.importances_mean,index=Xte.columns).sort_values(ascending=False).round(4))
p=res['grad. boosting']; te=test.assign(p=p)
te['decile']=pd.qcut(te.p.rank(method='first'),10,labels=False)+1
print(te.groupby('decile').churn_next_month.agg(['mean','size']).round(3).T)
pr,rc,_=precision_recall_curve(yte,p); plt.plot(rc,pr); plt.axhline(yte.mean(),ls='--',c='gray'); plt.xlabel('recall'); plt.ylabel('precision'); plt.title('PR curve (test)'); plt.show()
k=int(0.1*len(te)); top=te.nlargest(k,'p'); print(f'Top 10% by risk: precision {top.churn_next_month.mean():.2f}, lift {top.churn_next_month.mean()/yte.mean():.1f}x, captures {top.churn_next_month.sum()/yte.sum():.0%} of churners')

## 5. Robustness: rolling-origin check
Repeat the train-past / test-future procedure across several cutoffs to see whether performance is stable (winter months are expected to differ).

In [ ]:
rows=[]
for cut in ['2024-09','2024-12','2025-03','2025-06','2025-08']:
    c0=pd.Period(cut); a=df[df.month<=c0]; b=df[(df.month>c0+1)&(df.month<=c0+3)]
    m=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0).fit(prep(a),a.churn_next_month)
    pb=m.predict_proba(prep(b))[:,1]; rows.append((cut,len(b),b.churn_next_month.mean(),roc_auc_score(b.churn_next_month,pb),average_precision_score(b.churn_next_month,pb)))
print(pd.DataFrame(rows,columns=['train_until','n_test','test_churn','AUC','AP']).round(3))

## 6. Conclusions
* **`lifetime_months` is target leakage** (equals `tenure_months` iff the account churns) – excluded. With it the AUC is ~0.999; without it ~0.69.
* **Yes, but only modestly.** Leakage-free, the models reach ROC-AUC ≈ 0.69 (95 % CI ≈ 0.66–0.72) and PR-AUC ≈ 0.21–0.22 vs a 0.095 base rate on a held-out future period. The top 10 % riskiest accounts churn at ~26 % (≈2.8× lift, ~27 % of churners captured). Useful for prioritising outreach, not for confident individual predictions.
* **Signal:** support tickets (more → more churn) and usage hours (less → more churn). Plan, seats, fee and tenure add little. A boosted model does no better than logistic regression.
* **Seasonality / drift:** churn jumps to 40–55 % for Nov 2024–Jan 2025 snapshots; the data have a single winter, so we can't tell whether that is recurring seasonality or a one-off event. Performance and base rate vary by period (section 5); calibrate/monitor before relying on absolute probabilities.
* **Caveats:** ~half of test rows belong to accounts also in train; the bootstrap CI covers sampling noise only, not period-to-period shifts. Next steps: understand the winter spike, add history features (usage trend, ticket trend – available by lagging earlier snapshots), and calibrate probabilities.